# PINN Review — Part I: Papers 11–17 (2020–2021)

Executable reproductions for XPINN, NTK theory, DeepONet, NeuralPDE, Stiff-PINN and XPINN generalization bounds.

Source: Physics-Informed Neural Networks 2019–2026 review article.


## Shared Setup


In [ ]:
import math, time, random, warnings
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
torch.set_default_dtype(torch.float32)
SEED=7
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cpu'

def rel_l2(yhat,y):
    return float(torch.linalg.norm(yhat-y)/torch.linalg.norm(y))

class MLP(nn.Module):
    def __init__(self, din=1, dout=1, width=32, depth=3, dropout=0.0, act=nn.Tanh):
        super().__init__(); layers=[]; d=din
        for _ in range(depth):
            layers += [nn.Linear(d,width), act()]
            if dropout: layers += [nn.Dropout(dropout)]
            d=width
        layers += [nn.Linear(d,dout)]
        self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

def grad(y,x,order=1):
    g=y
    for _ in range(order):
        g=torch.autograd.grad(g,x,torch.ones_like(g),create_graph=True,retain_graph=True)[0]
    return g

def train(loss_fn, model, steps=600, lr=2e-3):
    opt=torch.optim.Adam(model.parameters(),lr=lr); hist=[]
    t=time.time()
    for i in range(steps):
        opt.zero_grad(); loss=loss_fn(); loss.backward(); opt.step()
        if i % max(1,steps//100)==0: hist.append(float(loss.detach()))
    return hist, time.time()-t

results=[]
def record(paper,method,error,seconds,notes):
    results.append(dict(paper=paper,method=method,relative_L2=error,seconds=seconds,notes=notes))


## Paper 11 — Jagtap & Karniadakis (2020): XPINN — Extended Domain Decomposition


In [ ]:
torch.manual_seed(SEED)
xa=xtrain[xtrain[:,0]<.5]; ya=target_piece(xa); xb2=xtrain[xtrain[:,0]>=.5]; yb2=target_piece(xb2)
a=MLP(width=10,depth=2); b2=MLP(width=30,depth=3); pars=list(a.parameters())+list(b2.parameters()); opt=torch.optim.Adam(pars,2e-3); st=time.time()
for _ in range(300):
    xi=torch.tensor([[.5]]); loss=((a(xa)-ya)**2).mean()+((b2(xb2)-yb2)**2).mean()+10*((a(xi)-b2(xi))**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
t=time.time()-st
with torch.no_grad(): yx=torch.where(xt<.5,a(xt),b2(xt))
e=rel_l2(yx,yt); record('Jagtap & Karniadakis XPINN','heterogeneous XPINN proxy',e,t,'small/large subnet')
plt.figure(figsize=(7,4)); plt.plot(xt,yt,'k',label='target'); plt.plot(xt,yx,'--',label='XPINN'); plt.legend(); plt.title('Heterogeneous subnetwork capacity'); plt.show()


## Paper 12 — Wang, Yu & Perdikaris (2020): NTK Perspective on PINN Failure


In [ ]:
torch.manual_seed(SEED)
ntk=MLP(width=24,depth=2); xb=torch.tensor([[0.],[1.]],requires_grad=True); xr=torch.linspace(0,1,20).reshape(-1,1).requires_grad_(True)
params=[p for p in ntk.parameters() if p.requires_grad]
def jac_rows(vals):
    rows=[]
    for val in vals.reshape(-1):
        gs=torch.autograd.grad(val,params,retain_graph=True,allow_unused=True)
        rows.append(torch.cat([(g if g is not None else torch.zeros_like(p)).reshape(-1) for g,p in zip(gs,params)]))
    return torch.stack(rows)
Jb=jac_rows(ntk(xb)); ur=ntk(xr); Jr=jac_rows(grad(ur,xr,2))
Kb=Jb@Jb.T; Kr=Jr@Jr.T; eb=torch.linalg.eigvalsh(Kb).detach().numpy(); er=torch.linalg.eigvalsh(Kr).detach().numpy(); ratio=float(er.max()/eb.max())
record('Wang et al. NTK','empirical NTK diagnostic',ratio,0.0,'value is residual/boundary max eigenvalue ratio')
plt.figure(figsize=(7,4)); plt.semilogy(np.sort(eb)[::-1]+1e-16,'o-',label='boundary block'); plt.semilogy(np.sort(er)[::-1]+1e-16,'o-',label='residual block'); plt.legend(); plt.title(f'Empirical PINN NTK spectra; scale ratio={ratio:.1f}'); plt.show()


## Papers 13–15 — Karniadakis et al. Review, DeepONet, NeuralPDE


In [ ]:
taxonomy=pd.DataFrame([
('Original PINN','learning'),('Adversarial UQ','learning + probabilistic'),('NN-aPC','observational + learning'),('DPINN','inductive + learning'),('DeepXDE/RAR','learning'),('PPINN','inductive + learning'),('VPINN','learning/variational'),('High-speed PINN','observational + learning'),('B-PINN','probabilistic learning'),('cPINN','inductive + learning'),('XPINN','inductive + learning'),('NTK weighting','learning/optimization'),('DeepONet','inductive/operator'),('NeuralPDE','learning/software'),('Stiff-PINN','inductive/model reduction'),('XPINN theory','generalization theory')],columns=['study','bias pathway'])
taxonomy


## Paper 16 — Ji et al. (2021): Stiff-PINN — QSSA Reduction for Stiff Kinetics


In [ ]:
torch.manual_seed(SEED)
m=30; sensors=torch.linspace(0,1,m)
class DeepONet(nn.Module):
    def __init__(self,p=32): super().__init__(); self.branch=MLP(m,p,48,2); self.trunk=MLP(1,p,48,2); self.bias=nn.Parameter(torch.zeros(1))
    def forward(self,u,y): return (self.branch(u)*self.trunk(y)).sum(1,keepdim=True)+self.bias
D=DeepONet(); nfun=140; U=[]; Y=[]; O=[]
for _ in range(nfun):
    a=torch.randn(4)*.5; u=sum(a[k]*torch.sin((k+1)*math.pi*sensors) for k in range(4)); y=torch.rand(6,1); out=sum(a[k]*(1-torch.cos((k+1)*math.pi*y))/((k+1)*math.pi) for k in range(4)); U.append(u.repeat(6,1)); Y.append(y); O.append(out)
U=torch.cat(U); Y=torch.cat(Y); O=torch.cat(O); h,t=train(lambda:((D(U,Y)-O)**2).mean(),D,400)
a=torch.tensor([.4,-.3,.2,.1]); u=sum(a[k]*torch.sin((k+1)*math.pi*sensors) for k in range(4)); yg=torch.linspace(0,1,200).reshape(-1,1); oo=sum(a[k]*(1-torch.cos((k+1)*math.pi*yg))/((k+1)*math.pi) for k in range(4)); pp=D(u.repeat(len(yg),1),yg).detach(); e=rel_l2(pp,oo); record('Lu et al. DeepONet','DeepONet',e,t,'antiderivative operator')
plt.figure(figsize=(7,4)); plt.plot(yg,oo,label='exact operator'); plt.plot(yg,pp,'--',label='DeepONet'); plt.legend(); plt.title(f'Unseen input function, relative L2={e:.2e}'); plt.show()


## Paper 17 — Hu et al. (2021): When XPINNs Improve Generalization


In [ ]:
from torch.quasirandom import SobolEngine
true=float(torch.trapz(torch.exp(-200*(torch.linspace(0,1,20000)-.67)**2),torch.linspace(0,1,20000)))
Ns=[16,32,64,128,256,512]; em=[]; eq=[]
for n in Ns:
    vals=[]
    for _ in range(30):
        x=torch.rand(n); vals.append(float(torch.exp(-200*(x-.67)**2).mean()))
    em.append(np.mean(np.abs(np.array(vals)-true)))
    x=SobolEngine(1,scramble=True,seed=SEED).draw(n).squeeze(); eq.append(abs(float(torch.exp(-200*(x-.67)**2).mean())-true))
record('Zubov et al. NeuralPDE','Sobol integration diagnostic',eq[-1],0.0,'integral absolute error')
plt.figure(figsize=(7,4)); plt.loglog(Ns,em,'o-',label='Monte Carlo'); plt.loglog(Ns,eq,'o-',label='Sobol'); plt.legend(); plt.xlabel('points'); plt.ylabel('absolute integration error'); plt.title('Loss discretization strategies'); plt.show()
